In [ ]:
###### Create Engine #####
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# load environment variable from .env
load_dotenv()


db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)

In [ ]:
##### DataBase initialisation
from sqlalchemy import text

print("Initializing OLTP Schema and Tables...")

with engine.connect() as conn:
    # Create the Operational Schema
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS oltp;"))
    
    # Define Normalized OLTP Tables

    # Customers Table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS oltp.customers (
            customerid INT PRIMARY KEY,
            country VARCHAR(100)
        );
    """))
    
    # Products Table
    # Added a CHECK constraint on unitprice as required by the project
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS oltp.products (
            stockcode VARCHAR(50) PRIMARY KEY,
            description TEXT,
            unitprice NUMERIC CHECK (unitprice >= 0)
        );
    """))
    
    # Orders Table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS oltp.orders (
            invoiceno VARCHAR(50) PRIMARY KEY,
            invoicedate TIMESTAMP,
            customerid INT REFERENCES oltp.customers(customerid)
        );
    """))
    
    # Order_Items Table
    # Added a CHECK constraint to ensure quantity is not zero (allows negatives for returns)
    conn.execute(text("""
        DROP TABLE oltp.order_items;
        CREATE TABLE IF NOT EXISTS oltp.order_items (
            transaction_id INT PRIMARY KEY,
            invoiceno VARCHAR(50) REFERENCES oltp.orders(invoiceno),
            stockcode VARCHAR(50) REFERENCES oltp.products(stockcode),
            quantity INT CHECK (quantity >= 0)
        );
    """))
    
    conn.commit()
    print("OLTP Tables created successfully.")

In [ ]:
#### Loading the Data into database

print("Loading data from staging table to OLTP tables...")

with engine.connect() as conn:
    # Insert Data into Customers
    # Using DISTINCT to avoid duplicate customer records
    conn.execute(text("""
        INSERT INTO oltp.customers (customerid, country)
        SELECT DISTINCT customerid, country
        FROM retail_data_clean
        WHERE customerid IS NOT NULL
        ON CONFLICT (customerid) DO NOTHING;
    """))


    # inserting into products
    # Using DISTINCT ON to resolve PrimeMart's duplicate product descriptions issue
    # This grabs the most recent description/price for each unique stockcode
    conn.execute(text("""
        INSERT INTO oltp.products (stockcode, description, unitprice)
        SELECT DISTINCT ON (stockcode) stockcode, description, unitprice
        FROM retail_data_clean
        WHERE stockcode IS NOT NULL
        ORDER BY stockcode, invoicedate DESC
        ON CONFLICT (stockcode) DO NOTHING;
    """))

    # Insert into Orders table
    conn.execute(text(
        """
        INSERT INTO oltp.orders (invoiceno,invoicedate,customerid)
        SELECT DISTINCT invoiceno,invoicedate,customerid
        FROM retail_data_clean
        ON CONFLICT (invoiceno) DO NOTHING;
        """
    ))

    # Insert into Order_items table
    conn.execute(text(
        """
        INSERT INTO oltp.order_items (transaction_id, invoiceno, stockcode, quantity )
        SELECT DISTINCT transaction_id, invoiceno, stockcode, quantity
        FROM retail_data_clean
        ON CONFLICT (transaction_id) DO NOTHING;
        """
    ))

    conn.commit()
    print('data inserted successfully')

In [ ]:
# Creating Indexes for faster query